# データセットの結合を手で通してみる

**`libero_combined_bowl5_initpose_l90clean` を組む手順を、小さいデータで一通り練習する。**
使うスクリプトも、打つコマンドの形も、本番とまったく同じである。**違うのは本数だけ。**

## これで分かること

| | |
|---|---|
| **外す** | 台帳で選んだ本を落とす（`rebuild_dataset.py`） |
| **抜く** | 別のデータから指定した本だけを取り出す（同じスクリプト） |
| **繋ぐ** | 2 つを 1 つにする（`merge_many.py`） |
| **答え合わせ** | 行動列の sha256 で 1 本ずつ突き合わせる（`compare_action_signatures.py`） |
| **落とし穴** | **繋いだ本は末尾に付くので、番号では対応が取れない** |

## これでは分からないこと

**本番の 21,642 本は Colab では組めない。**

| | 要る量 |
|---|---|
| 部品を落とす | 約 30 GB |
| 親を組む | 26 GB |
| 出来上がり | 26 GB |

**Colab の空きは 80 GB 前後で、入らない。**本番は手元のマシンか、ディスクの大きい台で回すこと
（[手順書](https://huggingface.co/datasets/siruku6/libero_plus_bowl5_initpose/tree/main/rebuild)）。

**練習に使うのは 397 本・0.34 GB のデータなので、全部で 10 分ほどで終わる。**

## 1. ランタイムを確認する

**結合だけなら GPU は要らない。**最後の学習の節だけ GPU を使う。

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/content/lerobot_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU なし。結合までは通るが、最後の学習の節は飛ばすこと。")

## 2. システムパッケージを準備する

In [ ]:
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


run_quiet(["apt-get", "update", "-qq"])
run_quiet(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. LeRobot を入れる

**`merge_many.py` が `lerobot.datasets` を使う。**ここが唯一の重い依存で、
`rebuild_dataset.py` のほうは numpy・pyarrow・pandas だけで動く。

In [ ]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ]
)

smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 4. 組み上げの道具を取り寄せる

**スクリプトは Hugging Face のデータセットに置いてある。**
`libero_plus_bowl5_initpose` の `rebuild/` に、手順・ソース一式・台帳が入っている。

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

WORK = Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)

BUNDLE_REPO = "siruku6/libero_plus_bowl5_initpose"

bundle_files = [
    name
    for name in __import__("huggingface_hub").HfApi().list_repo_files(
        BUNDLE_REPO, repo_type="dataset"
    )
    if name.startswith("rebuild/solution_") and name.endswith(".tar.gz")
]

if len(bundle_files) != 1:
    raise RuntimeError(f"ソース一式が一意に決まらない: {bundle_files}")

tarball = hf_hub_download(
    BUNDLE_REPO,
    bundle_files[0],
    repo_type="dataset",
    local_dir=str(WORK / "bundle"),
)

run_quiet(["tar", "xzf", tarball, "-C", str(WORK)])

REPO = WORK / "solution_src"
REBUILD = REPO / "solution/tools/contact_clean/rebuild_dataset.py"
MERGE = REPO / "solution/tools/regen/merge_many.py"
COMPARE = REPO / "solution/tools/regen/compare_action_signatures.py"

for path in (REBUILD, MERGE, COMPARE):
    if not path.exists():
        raise RuntimeError(f"入っていない: {path}")

print(f"道具を展開した: {REPO}")

## 5. 練習用の部品を落とす

**`reverse_lerobot_v2`（397 本・3 課題・0.34 GB）を使う。**
小さくて、本番と同じ LeRobot v3.0 の形をしている。

In [ ]:
ORIGINAL = WORK / "original"

snapshot_download(
    "siruku6/reverse_lerobot_v2",
    repo_type="dataset",
    local_dir=str(ORIGINAL),
)

info = json.loads((ORIGINAL / "meta/info.json").read_text())
print(f"{info['codebase_version']} / {info['total_episodes']} 本 "
      f"/ {info['total_frames']} コマ / {info['total_tasks']} 課題")

## 6. 「外す」を通す

**台帳は「残す本の番号の並び」である。**ここでは**先頭 40 本**を外してみる。

**なぜ先頭か。**末尾を外して末尾に足し戻すと並びが元どおりになってしまい、
**この手順のいちばん大事な癖が見えない。**本番で外す 250 本はデータの真ん中にあり、
足し直す 250 本は末尾に付く。

**動画は作り直さずハードリンクで置く**ので、ディスクはほとんど増えない。
LeRobot v3.0 が「1 本の mp4 に何十本も詰めて、開始時刻で切り出して読む」形だからである。

In [ ]:
TOTAL = int(info["total_episodes"])
DROP = 40

# part_a = 先頭 40 本を外した残り / part_b = その 40 本
(WORK / "keep_rest.json").write_text(json.dumps({"episodes": list(range(DROP, TOTAL))}))
(WORK / "keep_head.json").write_text(json.dumps({"episodes": list(range(DROP))}))

PART_A = WORK / "part_a"
shutil.rmtree(PART_A, ignore_errors=True)

print(run_quiet([sys.executable, str(REBUILD),
                 "--root", str(ORIGINAL), "--output", str(PART_A),
                 "--keep-file", str(WORK / "keep_rest.json")]).stdout)

## 7. 「抜く」を通す

**同じスクリプトである。**残す番号を変えるだけで、「外す」にも「抜く」にもなる。

In [ ]:
PART_B = WORK / "part_b"
shutil.rmtree(PART_B, ignore_errors=True)

print(run_quiet([sys.executable, str(REBUILD),
                 "--root", str(ORIGINAL), "--output", str(PART_B),
                 "--keep-file", str(WORK / "keep_head.json")]).stdout)

## 8. 「繋ぐ」を通す

**ここだけが lerobot を要る。**`--base` の末尾に `--add` を足していく。

**本番との違いは本数だけである。**同じコマンドで 21,642 本を繋いでいる。

In [ ]:
MERGED = WORK / "merged"
shutil.rmtree(MERGED, ignore_errors=True)

merge_env = os.environ.copy()
merge_env["PYTHONPATH"] = str(LEROBOT_SRC) + os.pathsep + merge_env.get("PYTHONPATH", "")

result = subprocess.run(
    [sys.executable, str(MERGE),
     "--base", str(PART_A), "--base-repo-id", "local/part_a",
     "--add", str(PART_B),
     "--output", str(MERGED), "--output-repo-id", "local/merged"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=merge_env,
)
print(result.stdout[-3000:])

if result.returncode != 0:
    raise RuntimeError("繋げなかった")

## 9. 答え合わせ——**番号では合わない**

**元のデータと同じ 397 本が揃っているはずである。**確かめてみる。

**まず番号どおりに比べる。**外した 40 本は末尾に付け直されているので、**1 本も合わない。**
**壊れているのではなく、並びが変わっただけである。**

これが本番でも起きることで、
[台帳](https://huggingface.co/datasets/siruku6/libero_plus_bowl5_initpose/blob/main/rebuild/manifests/l90clean_replace.json)
が `source_episode_index` を持っている理由でもある。

In [ ]:
print("--- 番号どおりに比べる ---")
print(subprocess.run([sys.executable, str(COMPARE),
                      "--a", str(ORIGINAL), "--b", str(MERGED)],
                     text=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT).stdout)

print("--- 番号を見ずに、集合として比べる ---")
print(subprocess.run([sys.executable, str(COMPARE),
                      "--a", str(ORIGINAL), "--b", str(MERGED), "--unordered"],
                     text=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT).stdout)

**番号どおりでは `0 / 397`、番号を見なければ `397 / 397` になる。**

| 比べ方 | 結果 | 意味 |
|---|---|---|
| 番号どおり | **0 / 397** | 並びが変わった |
| 番号を見ない | **397 / 397** | **中身は 1 本残らず同じ** |

**本番の突き合わせでこれを取り違えないこと。**
`l90clean` の検証で `21642 / 21642` が出るのは、**足した 250 本の位置まで台帳どおりに
再現したからである。**

## 10. 本番と同じ形——台帳で入れ替える

**`l90clean` の後段は、いまやったことの組み合わせである。**

1. **親**から「汚れた本」を外す
2. **別の部品**から「綺麗な本」を抜く
3. 末尾に繋ぐ

**台帳がその 3 つを決めている。**本番の台帳はこういう形をしている。

```json
{
  "dropped_episode_indices": [17784, 17791, ...],
  "appended_episodes": [{"source_episode_index": 3661, ...}, ...]
}
```

**`source_episode_index` は「部品の何番から取ったか」である。**
同じ形の小さな台帳を書いて、通してみる。

**答えが分かる形にしておく**——部品として「元データから 10 本抜いたもの」を用意し、
親からはその同じ 10 本を外す。**正しく通れば、中身は元データと一致するはずである。**

In [ ]:
REPLACE = 10
TARGET = list(range(100, 100 + REPLACE))   # 入れ替える 10 本（データの真ん中）

manifest = {
    "dropped_episode_indices": TARGET,
    "appended_episodes": [{"source_episode_index": i} for i in TARGET],
}
(WORK / "practice_replace.json").write_text(json.dumps(manifest, indent=2))

# 部品を作る（本番では siruku6/libero_90_clean に当たるもの）
POOL = WORK / "pool"
shutil.rmtree(POOL, ignore_errors=True)
(WORK / "keep_pool_src.json").write_text(json.dumps({"episodes": TARGET}))
run_quiet([sys.executable, str(REBUILD), "--root", str(ORIGINAL),
           "--output", str(POOL), "--keep-file", str(WORK / "keep_pool_src.json")])

# 台帳から keep-file を作る（rebuild_l90clean.sh の 1 段目と同じ）
dropped = set(manifest["dropped_episode_indices"])
kept = [i for i in range(TOTAL) if i not in dropped]
sources = [row["source_episode_index"] for row in manifest["appended_episodes"]]
pool_rows = {original: new for new, original in enumerate(TARGET)}

(WORK / "keep_parent.json").write_text(json.dumps({"episodes": kept}))
(WORK / "keep_pool.json").write_text(
    json.dumps({"episodes": [pool_rows[i] for i in sources]}))

print(f"親から残す {len(kept)} 本 / 部品から足す {len(sources)} 本")

In [ ]:
PARENT_KEPT = WORK / "parent_kept"
REPLACEMENTS = WORK / "replacements"
REBUILT = WORK / "rebuilt"

for path in (PARENT_KEPT, REPLACEMENTS, REBUILT):
    shutil.rmtree(path, ignore_errors=True)

run_quiet([sys.executable, str(REBUILD), "--root", str(ORIGINAL),
           "--output", str(PARENT_KEPT), "--keep-file", str(WORK / "keep_parent.json")])
run_quiet([sys.executable, str(REBUILD), "--root", str(POOL),
           "--output", str(REPLACEMENTS), "--keep-file", str(WORK / "keep_pool.json")])

result = subprocess.run(
    [sys.executable, str(MERGE),
     "--base", str(PARENT_KEPT), "--base-repo-id", "local/parent_kept",
     "--add", str(REPLACEMENTS),
     "--output", str(REBUILT), "--output-repo-id", "local/rebuilt"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=merge_env,
)
print(result.stdout[-1200:])

print("--- 元データと突き合わせる（番号は見ない）---")
print(subprocess.run([sys.executable, str(COMPARE),
                      "--a", str(ORIGINAL), "--b", str(REBUILT), "--unordered"],
                     text=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT).stdout)

**`397 / 397` が出れば、台帳どおりに組み直せている。**

**本番との違いは 3 つだけである。**

| | 練習 | 本番 |
|---|---|---|
| 親 | 元データ 397 本 | `libero_combined_bowl5_initpose` 21,642 本 |
| 部品 | 元データから抜いた 10 本 | `libero_90_clean` 3,911 本 |
| 入れ替え | 10 本 | **250 本** |

**本番では、外す本と足す本が別物である**（汚れた手本を綺麗な手本に替える）ので、
コマ数が変わる——**−5,803 コマ**だった。**本数と課題数は変わらない。**

## 11. 繋いだデータで学習できるか

**ここから GPU が要る。**結合を確かめるだけなら、この節は飛ばしてよい。

**目的は「良い模型を作ること」ではなく、「繋いだデータが学習に通ること」を確かめること**である。
300 歩だけ回す。

In [ ]:
if not HAS_GPU:
    raise RuntimeError("GPU ランタイムに切り替えてから実行すること。")

BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = "7bb70aa5bc92b82c9239142775d3a173103567ff"
VLM_REPO = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

BASE_MODEL_LOCAL = snapshot_download(BASE_MODEL_REPO, revision=BASE_MODEL_REVISION)
OUTPUT_DIR = Path("/content/outputs/merge_practice")
STEPS = 300

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    "--policy.optimizer_lr=3e-4",
    "--dataset.repo_id=local/merged",
    f"--dataset.root={MERGED}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=merge_practice",
    f"--steps={STEPS}",
    "--batch_size=1",
    "--num_workers=0",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    "--save_checkpoint=false",
    "--log_freq=50",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    "--peft.r=16",
    "--peft.lora_alpha=16",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = str(LEROBOT_SRC) + os.pathsep + training_env.get("PYTHONPATH", "")
training_env["ACCELERATE_MIXED_PRECISION"] = (
    "bf16" if torch.cuda.is_bf16_supported() else "fp16"
)
training_env["PYTHONUNBUFFERED"] = "1"
training_env["TQDM_DISABLE"] = "1"

process = subprocess.Popen(command, cwd=LEROBOT_DIR, env=training_env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()
    if "step:" in line and "loss:" in line:
        print(line)

if process.wait() != 0:
    raise RuntimeError("学習が落ちた")

print("繋いだデータで学習が回った。")

**loss が出れば、繋いだデータは学習に通っている。**
数字の良し悪しは 300 歩では読めないので、そこは見ない。

## 12. 本番に上がるときに変わること

| | 練習 | 本番 |
|---|---|---|
| 本数 | 397 | **21,642** |
| 入れ替える本 | 10 | **250** |
| ディスク | 1 GB 以下 | **120 GB** |
| 繋ぐ時間 | 数秒 | **約 3 分** |
| メモリ | 気にしない | **Docker に 8 GB 以上** |
| 走らせる所 | Colab | **ディスクのある台** |

**コマンドの形は変わらない。**本番では `build_initpose_datasets.sh` と
`rebuild_l90clean.sh` が、いまここで打ったものを順に呼んでいるだけである。

**手順書はここにある。**
<https://huggingface.co/datasets/siruku6/libero_plus_bowl5_initpose/tree/main/rebuild>